## 0. 환경 설정

In [ ]:
!pip install neo4j-graphrag neo4j openai

API 로그인 > API Key 생성 : https://openai.com/chatgpt/

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "s"

## 1. GraphDB 불러오기

Neo4j Sandbox : https://sandbox.neo4j.com/

### 1-1. Neo4j 드라이버 설정

In [ ]:
from neo4j import GraphDatabase, basic_auth
import openai

driver = GraphDatabase.driver(
  "neo4j://35.175.117.100:7687",
  auth=basic_auth("neo4j", "lessons-suppressions-lifts"))

```cypher
MATCH (m:Movie {title:$movie})<-[:RATED]-(u:User)-[:RATED]->(rec:Movie)
RETURN distinct rec.title AS recommendation LIMIT 20
```

- `(m:Movie {title:$movie})` : title이 $movie 인 노드
- `(rec:Movie)` : 이 노드는 rec이라는 변수로 지정
- `RETURN distinct rec.title` : rec 변수에 있는 노드의 title을 RETURN



In [ ]:
cypher_query = '''
MATCH (m:Movie {title:$movie})<-[:RATED]-(u:User)-[:RATED]->(rec:Movie)
RETURN distinct rec.title AS recommendation LIMIT 20
'''

with driver.session(database="neo4j") as session:
  results = session.execute_read(
    lambda tx: tx.run(cypher_query,
                      movie="Crimson Tide").data())
  for record in results:
    print(record['recommendation'])

#driver.close()

## 2. Plot(줄거리) 임베딩 추가

- Retriever 목표 : 줄거리 내용으로 검색해서 비슷한 내용의 영화 찾기

How? 벡터 검색

### 2-1. 텍스트 to 임베딩 벡터 (OpenAI 임베딩 모델)

In [ ]:
import os
import openai

def generate_embedding(text):
    # Initialize the OpenAI client explicitly with the API key
    client = openai.OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    embedding = client.embeddings.create(input = [text], model='text-embedding-ada-002').data[0].embedding
    return embedding

In [ ]:
def add_embedding_to_movie(tx):
    """모든 Movie 노드의 plot을 임베딩하고, 그 값을 embedding 속성에 추가"""
    result = tx.run("MATCH (m:Movie) WHERE m.plot IS NOT NULL RETURN m.title AS title, m.plot AS plot, ID(m) AS id LIMIT 100")
    cnt = 0
    for record in result:
        cnt += 1
        print(cnt)
        title = record["title"]
        plot = record["plot"]
        node_id = record["id"]
        print(plot)
        print("=================")
        embedding = generate_embedding(plot)
        # 임베딩 벡터를 Neo4j에 저장
        tx.run("MATCH (m:Movie) WHERE ID(m) = $id SET m.plotEmbedding = $embedding", id=node_id, embedding=embedding)
        print(f"Updated movie '{title}' with embedding.")

with driver.session() as session:
    session.execute_write(add_embedding_to_movie)

## 3. Vector INDEX 로 Retriever 생성하기

Embedding 모델 : https://platform.openai.com/docs/models/embeddings

VECTOR INDEX : https://neo4j.com/docs/cypher-manual/current/indexes/semantic-indexes/vector-indexes/

인덱스(VECTOR INDEX) 생성하기
```cypher
CREATE VECTOR INDEX moviePlotsEmbedding FOR (n:Movie) ON (n.plotEmbedding) OPTIONS {indexConfig: {
 `vector.dimensions`: 1536,
 `vector.similarity_function`: 'cosine'
}}
```

In [ ]:
from neo4j_graphrag.retrievers import VectorRetriever
from neo4j_graphrag.embeddings.openai import OpenAIEmbeddings
embedder = OpenAIEmbeddings(model="text-embedding-ada-002")
retriever = VectorRetriever(
    driver,
    index_name="moviePlotsEmbedding",
    embedder=embedder,
    return_properties=["title", "plot"],
)

In [ ]:
query_text = "A cowboy doll is jealous when a new spaceman figure becomes the top toy." # 토이스토리 줄거리
retriever_result = retriever.search(query_text=query_text, top_k=3)
print(retriever_result)

In [ ]:
retriever_result

In [ ]:
query_text = "A movie about a shooting incident."
retriever_result = retriever.search(query_text=query_text, top_k=3)
print(retriever_result)

In [ ]:
print(retriever_result)

In [ ]:
for k, item in enumerate(retriever_result.items):
    print(f"TOP {k+1}")
    print(item.content)
    print("score : ", item.metadata["score"])